# Fisher-vs-MCMC cross-check: PFS $P_\ell + B_0$ + DESI DR2 BAO + BBN+$n_s$ ($\Lambda$CDM)

BlackJAX NUTS posterior on the **same forecast likelihood** as
`fisher_joint_PFS_BAO_BBN_ns_LCDM.ipynb`, to test whether the Fisher
(Gaussian) forecast is consistent with the full posterior.

**Setup (matches the Fisher notebook exactly):**
- PFS full-shape $P_\ell(k) + B_0$ (7 bins, AP), DESI DR2 BAO, BBN + $n_{s,10}$ + EFT/stochastic priors.
- **Noiseless mock**: the "data" is the fiducial theory vector, so the posterior is centred at the
  fiducial and its width/shape directly test the Fisher approximation.
- **Sample the full varied block** (cosmology + per-bin EFT/survey params, minus the fixed ones) —
  the same block the Fisher marginalises — then compare the marginal cosmology posterior to the
  Fisher cosmology block.
- **Priors centred at the fiducial** with the Fisher $\sigma$'s (self-consistent forecast).
- **Whitening** uses the prior-included Fisher (the EFT directions are prior-dominated).

> Runtime: the P+B theory ($n_\mu=n_\phi=65$, 7 bins) is evaluated on every gradient step, so a
> production chain is expensive (hours). `SMOKE_TEST=True` runs a tiny chain first to validate the
> pipeline end-to-end in minutes; set it to `False` for a science run.

In [ ]:
from functools import partial
import os
import time

N_THREADS = os.cpu_count() or 1
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = str(N_THREADS)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-jaxptpolypol")

import matplotlib
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
plt.rcParams.update({
    "text.usetex": True, "font.family": "sans-serif",
    "font.sans-serif": "Computer Modern", "font.size": 22})

import numpy as np
from tqdm.auto import tqdm

import jax
jax.config.update("jax_enable_x64", True)

# Persistent XLA compilation cache: repeated notebook runs reuse compiled graphs
import pathlib as _pathlib
jax.config.update("jax_compilation_cache_dir", str(_pathlib.Path.home() / ".jax_xla_cache"))
jax.config.update("jax_persistent_cache_min_compile_time_secs", 10)
import jax.numpy as jnp
from jax.scipy.linalg import inv

from ps_1loop_jax import background as bg

from jaxptpolypol import (
    split_marginal_indices, make_constant_prior_fns, bin_lin_slices,
    make_marginal_log_posterior_perbin,
)
from jaxptpolypol.model import BispectrumTreeModel, CosmoEmulator, PS1LoopModel
from jaxptpolypol.params import CosmoParams, FullShapeSurveyParams, pack_joint_params
from jaxptpolypol.theory import (
    build_bispectrum_triangles_from_k_grid, compute_fiducial_distances,
    make_gaussian_joint_covariance_fn, make_joint_pk_bk_fn,
    make_joint_pk_bk_bin_fn,
)
from jaxptpolypol.bao import (
    load_desi_dr2, bao_fisher_matrix, add_bao_to_fullshape_fisher, make_bao_theory_fn,
)
from jaxptpolypol.priors import load_prior_spec, build_prior_sigmas_from_spec
from jaxptpolypol.inference import (
    fisher_matrix, marginalize_fisher, marginalized_fisher_block,
    fixed_and_varied_indices, gaussian_prior_fisher,
)
from jaxptpolypol.plotting import plot_contours, plot_Gaussian
from jaxptpolypol.chain_analysis import (
    make_fullshape_spec, packed_samples_to_dict, scalar_summary_packed, plot_trace,
)
from jaxptpolypol.sampler import (
    make_transform, make_full_params_fn, make_gaussian_log_prior,
    make_log_posterior, run_nuts, run_rwmh, samples_to_physical,
)


## Configuration (mirrors the Fisher notebook)

In [ ]:
# ── Unified fiducial cosmology (Planck 2018 TT,TE,EE+lowE+lensing+BAO) ──
FIDUCIAL = {
    'ombh2': 0.02242, 'omch2': 0.11933, 'logA': 3.047,
    'ns': 0.9665, 'h': 0.6766, 'tau': 0.0561,
}
MNU_FIXED = 0.06  # eV, not varied in LCDM

# ── PFS survey config ──
z_bins   = (0.7,  0.9,  1.1,  1.3,  1.5,  1.8,  2.2)
V_bins   = tuple(v * 1000.**3 for v in (0.59, 0.79, 0.96, 1.09, 1.19, 2.58, 2.71))
knl_bins = (0.52, 0.65, 0.82, 1.02, 1.29, 1.82, 2.88)
n_bar    = (3.06e-4, 9.61e-4, 9.75e-4, 6.54e-4, 3.40e-4, 2.02e-4, 3.51e-4)
n_zbins  = len(z_bins)

# ── Bispectrum / joint P+B config (identical to the Fisher notebook) ──
K_PK_MIN, K_PK_MAX, N_K = 0.02, 0.20, 37
K_BK_MIN, K_BK_MAX = 0.02, 0.08
K_NL_RSD = 0.45
NUM_MU = NUM_PHI = 65
N_GL = 16
BB_POWER_MODEL = 'kaiser'
BACKGROUND_MODE = 'direct'

PFS_EMULATOR = '/Users/nguyenmn/cosmopower-jax-for-pfs/cosmology/jense2024/jense_2023_camb_lcdm/networks/jense_2023_camb_lcdm_Pk_lin.npz'
BAO_DATA_DIR = "../../ext_data/bao_data/desi_bao_dr2"

# ── Forecast (cosmology) basis and priors ──
SHARED_KEYS = ('ombh2', 'omch2', 'logA', 'ns', 'h')
N_SHARED = len(SHARED_KEYS)
SHARED_LABELS = (r'$\omega_b$', r'$\omega_c$', r'$\log A$', r'$n_s$', r'$h$')
FIDUCIAL_SHARED = np.array([FIDUCIAL[k] for k in SHARED_KEYS])
COSMO_PRIORS = {'ombh2': 0.00055, 'ns': 0.042}  # BBN + ns10 (arXiv:2411.12022)
MODEL_TITLE = r'$\Lambda$CDM'

# ── MCMC run mode ──
SMOKE_TEST = True   # tiny chain to validate the pipeline end-to-end; set False for production
# Gradient-free random-walk MH (arXiv:2511.20757 uses MH; NUTS grad of the
# marginal posterior is a second-order XLA graph that exhausts memory).
if SMOKE_TEST:
    NUM_SAMPLES_MH, NUM_BURN_MH, NUM_CHAINS, SCAN_CHUNK_MH = 2000, 500, 1, 500
else:
    NUM_SAMPLES_MH, NUM_BURN_MH, NUM_CHAINS, SCAN_CHUNK_MH = 150_000, 15_000, 2, 5_000

## Emulator, models, fiducial parameters

In [ ]:
pklin_emulator = CosmoEmulator(probe='custom_log', emulator_path=PFS_EMULATOR)
ps1loop_model = PS1LoopModel(do_irres=True)
bispectrum_model = BispectrumTreeModel(do_AP=True, k_nl_rsd=K_NL_RSD)

cosmo_dict = {
    'ombh2': FIDUCIAL['ombh2'], 'omch2': FIDUCIAL['omch2'],
    'logA':  FIDUCIAL['logA'],  'ns':    FIDUCIAL['ns'], 'h': FIDUCIAL['h'],
    'z': 0.7, 'A_b': 3.13, 'eta_b': 0.603, 'logT_AGN': 7.8,
}
cosmo = CosmoParams(cosmo_dict)

# ── Bias / counterterm fiducials (arXiv:1907.06666) ──
def b1z(z): return 0.9 + 0.4 * z
def b2z(z): return -0.704 - 0.208 * z + 0.183 * z**2 - 0.00771 * z**3
def bG2z(z): return -(2. / 7.) * (b1z(z) - 1.)
def bGamma3z(z): return (23. / 42.) * (b1z(z) - 1.)
def Dplusz(z):
    return float(bg.growth_factor(
        cosmo_dict['ombh2'], cosmo_dict['omch2'], cosmo_dict['h'], z, mnu=MNU_FIXED))
def c0z(z): return 25. * Dplusz(z)**2
def c2z(z): return 25. * Dplusz(z)**2
def c4z(z): return Dplusz(z)**2

surveys = []
for z, knl, nd in zip(z_bins, knl_bins, n_bar):
    surveys.append(FullShapeSurveyParams(
        shared={'bias': {'b1': b1z(z), 'b2': b2z(z), 'bG2': bG2z(z), 'bGamma3': bGamma3z(z)},
                'stoch': {'P_shot': 1.0}, 'k_nl': knl, 'ndens': nd},
        pk={'ctr': {'c0': c0z(z), 'c2': c2z(z), 'c4': c4z(z), 'cfog': knl**(-4)},
            'stoch': {'a0': 0., 'a2': 0.}},
        bk={'ctr': {'c1': 0.0}, 'stoch': {'B_shot': 1.0, 'A_shot': 1.0}},
    ))
joint_survey_keys = surveys[0].joint_param_keys
packed_params = pack_joint_params(cosmo, surveys)
n_cosmo_params = sum(cosmo.param_sizes)
n_survey_params = len(joint_survey_keys)
Hz_fid, DAz_fid = compute_fiducial_distances(cosmo, z_bins)
print(f"Packed params: {packed_params.shape[0]} ({n_cosmo_params} cosmo + {n_survey_params}x{n_zbins} survey)")

## Joint P+B theory, covariance, and the data Fisher

In [ ]:
joint_fn = make_joint_pk_bk_fn(
    pklin_emulator=pklin_emulator, ps1loop_model=ps1loop_model, bispectrum_model=bispectrum_model,
    cosmo_keys=cosmo.param_keys, cosmo_sizes=cosmo.param_sizes, survey_keys=joint_survey_keys,
    ap=True, z_bins=z_bins, Hz_fid=Hz_fid, DAz_fid=DAz_fid,
    n_gl=N_GL, num_mu=NUM_MU, num_phi=NUM_PHI, background_mode=BACKGROUND_MODE)
joint_cov_fn = make_gaussian_joint_covariance_fn(
    pklin_emulator=pklin_emulator, ps1loop_model=ps1loop_model,
    cosmo_keys=cosmo.param_keys, cosmo_sizes=cosmo.param_sizes, survey_keys=joint_survey_keys,
    ap=True, z_bins=z_bins, Hz_fid=Hz_fid, DAz_fid=DAz_fid,
    bb_power_model=BB_POWER_MODEL, n_gl=N_GL, background_mode=BACKGROUND_MODE)
jitted_joint_fn = jax.jit(joint_fn)

k = jnp.linspace(K_PK_MIN, K_PK_MAX, N_K)
dk = float(k[1] - k[0]); n_k = int(k.shape[0])
triangles, triangle_dk = build_bispectrum_triangles_from_k_grid(k, k_min=K_BK_MIN, k_max=K_BK_MAX, dk=dk)
n_tri = int(triangles.shape[0])
print(f"P grid n_k={n_k}, dk={dk:.5f}; B triangles={n_tri}; per-bin block=3*{n_k}+{n_tri}")

jitted_joint_fn(packed_params, k=k, triangles=triangles).block_until_ready()  # warmup
%time jac = jax.jacfwd(jitted_joint_fn, argnums=0)(packed_params, k=k, triangles=triangles)
gauss_cov = joint_cov_fn(packed_params, V_survey=V_bins, k=k, dk=dk,
                         triangles=triangles, triangle_dk=triangle_dk)
F_pfs_full = fisher_matrix(gauss_cov, jac)
print("PFS P+B Fisher:", F_pfs_full.shape)

## DESI DR2 BAO: Fisher block + differentiable likelihood

In [ ]:
bao_dr2 = load_desi_dr2("all", data_dir=BAO_DATA_DIR)
fiducial_cosmo = packed_params[:n_cosmo_params]

# Fisher block (matches the Fisher notebook)
F_bao = bao_fisher_matrix(bao_dr2, fiducial_cosmo,
    cosmo_keys=cosmo.param_keys, cosmo_sizes=cosmo.param_sizes,)
F_pfs_bao_full = add_bao_to_fullshape_fisher(F_pfs_full, F_bao, n_cosmo=n_cosmo_params)

# Differentiable BAO theory over the packed cosmology block (for the MCMC likelihood)
bao_theory_fn = make_bao_theory_fn(bao_dr2,
    cosmo_keys=cosmo.param_keys, cosmo_sizes=cosmo.param_sizes, mnu_fixed=MNU_FIXED)
print("BAO data points:", len(bao_dr2.data_points), "| F_bao:", F_bao.shape)

## Priors: EFT/stochastic spec + BBN + $n_{s,10}$

In [ ]:
prior_spec = load_prior_spec('eft_eq12_2405_02252')
prior_sigmas = build_prior_sigmas_from_spec(
    cosmo_keys=cosmo.param_keys, cosmo_sizes=cosmo.param_sizes, survey_keys=joint_survey_keys,
    n_bins=n_zbins, observable='joint', spec=prior_spec, cosmo_priors=COSMO_PRIORS)
# bGamma3 ~ N(fid, 1^2) (arXiv:2511.20757 Table I): required proper prior for
# analytic marginalization; added to BOTH the Fisher and the MCMC sides.
bgamma3_off = joint_survey_keys.index(('shared', 'bias', 'bGamma3'))
for b in range(n_zbins):
    prior_sigmas[n_cosmo_params + b * n_survey_params + bgamma3_off] = 1.0
F_prior = gaussian_prior_fisher(packed_params.shape[0], prior_sigmas)
F_pfs_bao_prior_full = F_pfs_bao_full + F_prior
print(f"Prior entries: {len(prior_sigmas)} params (BBN ombh2, ns10 ns, EFT/stochastic survey)")

## Varied block, comparison Fisher, and whitening scales

In [ ]:
# Fixed (removed): cosmo z/A_b/eta_b/logT_AGN and per-bin k_nl, ndens.
# Sampled block theta_NL = varied cosmology + per-bin (b1, b2, bG2);
# the 11 linear EFT/stochastic params per bin are marginalized analytically
# (arXiv:2511.20757 Table I; see CONTEXT.md "c1 FoG counterterm").
fixed_cosmo = [5, 6, 7, 8]
split = split_marginal_indices(
    n_cosmo_params=n_cosmo_params, survey_keys=joint_survey_keys, n_bins=n_zbins,
    fixed_cosmo=fixed_cosmo,
    fixed_survey_keys={('shared', 'k_nl', None), ('shared', 'ndens', None)})
n_nl = split.n_nl
varied_idx = sorted(list(split.nl_idx) + list(split.lin_idx))   # same set as before

cosmo_varied_global = [i for i in range(n_cosmo_params) if i not in fixed_cosmo]
nl_pos = {full_idx: pos for pos, full_idx in enumerate(split.nl_idx)}
cosmo_nl_pos = [nl_pos[i] for i in cosmo_varied_global]         # cosmo positions in theta_NL

# Comparison Fisher: UNCHANGED — full-space Fisher + priors, marginalized to cosmology.
F_pfs_bao_prior_cosmo = marginalized_fisher_block(
    marginalize_fisher(F_pfs_bao_prior_full, varied_idx),
    [varied_idx.index(i) for i in cosmo_varied_global])

# Whitening for the SAMPLED block: Schur the prior-included Fisher down to theta_NL.
F_varied_prior = marginalize_fisher(F_pfs_bao_prior_full, varied_idx)
nl_in_varied = [varied_idx.index(i) for i in split.nl_idx]
F_nl_prior = marginalized_fisher_block(F_varied_prior, nl_in_varied)
fisher_sigma_nl = jnp.sqrt(jnp.diag(inv(F_nl_prior)))
fid_nl = packed_params[jnp.array(split.nl_idx)]
print(f"n_NL = {n_nl} ({len(cosmo_varied_global)} cosmo + {3 * n_zbins} bias), "
      f"n_lin marginalized = {split.n_lin}")


## Combined likelihood, whitening transform, and log-posterior

In [ ]:
# Combined (noiseless) forecast data vector: P+B ++ BAO, both at the fiducial.
pb_fid = jitted_joint_fn(packed_params, k=k, triangles=triangles)
bao_fid = bao_theory_fn(fiducial_cosmo)

# --- Exact per-bin factorization of the marginalized likelihood ---------------
# gauss_cov is block diagonal across redshift bins (covariance.py) and each bin's
# theory depends only on its own theta_lin, so the dense n_lin x n_lin
# marginalization splits into a sum of per-bin ones: identical posterior, ~n_zbins
# times smaller XLA graph. BAO carries no theta_lin dependence and enters once as
# a plain -0.5 r^T Cinv r term.
block_len = 3 * n_k + n_tri
bin_theory_kwargs = dict(
    pklin_emulator=pklin_emulator, ps1loop_model=ps1loop_model,
    bispectrum_model=bispectrum_model,
    cosmo_keys=cosmo.param_keys, cosmo_sizes=cosmo.param_sizes,
    survey_keys=joint_survey_keys,
    ap=True, z_bins=z_bins, Hz_fid=Hz_fid, DAz_fid=DAz_fid,
    n_gl=N_GL, num_mu=NUM_MU, num_phi=NUM_PHI, background_mode=BACKGROUND_MODE)
bin_theory_fns = [
    partial(make_joint_pk_bk_bin_fn(bin_index=b, **bin_theory_kwargs),
            k=k, triangles=triangles)
    for b in range(n_zbins)]
bin_blocks = [slice(b * block_len, (b + 1) * block_len) for b in range(n_zbins)]
bin_data = [pb_fid[sl] for sl in bin_blocks]
bin_cov_invs = [inv(gauss_cov[sl, sl]) for sl in bin_blocks]   # per-block inverse
bin_lin_idx = [split.lin_idx[sl] for sl in bin_lin_slices(split, n_zbins)]
bao_cov_inv = inv(jnp.asarray(bao_dr2.cov))

# --- Marginalized-block priors: fiducial-centered, Fisher-consistent widths ---
# (Stream A: same sigmas as the Fisher side; means at the fiducial so the
#  noiseless-mock posterior peaks at truth. Every lin param has a proper prior.)
mu_p = packed_params[jnp.array(split.lin_idx)]
sigma_p = jnp.array([prior_sigmas[i] for i in split.lin_idx])
prior_mean_fn, prior_sigma_fn = make_constant_prior_fns(mu_p, sigma_p)

# --- Sampled-block (theta_NL) Gaussian priors: whatever prior_sigmas assigns
#     to cosmo/bias slots (BBN ombh2, ns10, ...), fiducial-centered ---
nl_prior_entries = [
    (nl_pos[i], float(packed_params[i]), prior_sigmas[i])
    for i in split.nl_idx if i in prior_sigmas
]
log_prior_nl = make_gaussian_log_prior(n_nl, nl_prior_entries)

# Whitening + full-vector reconstruction for the sampled block only.
to_whitened, to_physical = make_transform(center=fid_nl, scale=fisher_sigma_nl)
full_params_fn = make_full_params_fn(packed_params, split.nl_idx)

log_post = make_marginal_log_posterior_perbin(
    bin_theory_fns=bin_theory_fns, bin_data=bin_data, bin_cov_invs=bin_cov_invs,
    bin_lin_idx=bin_lin_idx,
    extra_theory_fn=lambda p: bao_theory_fn(p[:n_cosmo_params]),
    extra_data=bao_fid, extra_cov_inv=bao_cov_inv,
    prior_mean_fn=prior_mean_fn, prior_sigma_fn=prior_sigma_fn,
    log_prior_nl_fn=log_prior_nl, to_physical=to_physical,
    full_params_fn=full_params_fn, include_logdet=True)

x0 = jnp.zeros(n_nl)


In [ ]:
# Forward gate: first (compiling) evaluation of the factorized log-posterior.
# The dense monolith is deliberately NOT built here for a cross-check -- that
# needs a second full n_zbins-bin compile; perbin == monolith at rtol=1e-10 is
# asserted by tests/test_marginal_perbin.py.
print(f"n_NL = {n_nl}, n_lin = {split.n_lin} "
      f"({split.n_lin // n_zbins}/bin), block_len = 3*{n_k}+{n_tri} = {block_len}")
print(f"per bin: data {tuple(bin_data[0].shape)}, cov_inv {tuple(bin_cov_invs[0].shape)}, "
      f"lin_idx {len(bin_lin_idx[0])} x {n_zbins} bins")
print(f"BAO: data {tuple(bao_fid.shape)}, cov_inv {tuple(bao_cov_inv.shape)}")

_t0 = time.perf_counter()
lp0 = float(jax.block_until_ready(log_post(x0)))
_dt = time.perf_counter() - _t0
# Noiseless mock + fiducial-centered priors => residual and NL-prior terms are 0;
# lp0 equals the (nonzero) -0.5*logdet(A Sigma_p) normalization. Record it.
print(f"log_post(x0) = {lp0:.6f}  (pure -0.5*logdet(A Sigma_p) term); "
      f"first call trace+compile+eval {_dt:.1f} s")
assert np.isfinite(lp0)


## Run random-walk MH (Fisher-whitened, isotropic 2.38/sqrt(d) proposal)

Gradient-free Metropolis-Hastings, as in the DESI DR1 reanalysis (arXiv:2511.20757, Montepython MH):
the marginal posterior needs only forward evaluations, avoiding the second-order XLA graph
that `grad(log_post)` (NUTS) requires, which exhausts memory on the 7-bin P+B model.

In [ ]:
%%time
rng_key = jax.random.key(42)
sample_bar = tqdm(total=NUM_CHAINS * NUM_SAMPLES_MH, desc="RWMH sampling", position=0)

def on_sample(chain_idx, num_chains, done, total):
    sample_bar.n = (chain_idx - 1) * total + done
    sample_bar.refresh()

try:
    samples_w, diagnostics = run_rwmh(
        rng_key, log_post, initial_position=x0,
        num_samples=NUM_SAMPLES_MH, num_chains=NUM_CHAINS,
        scan_chunk_size=SCAN_CHUNK_MH, sample_progress_fn=on_sample)
finally:
    sample_bar.close()

samples_w = samples_w[:, NUM_BURN_MH:, :]      # discard burn-in (no warmup phase in RWMH)
print("samples (post burn-in):", samples_w.shape)
print("acceptance per chain:", np.asarray(diagnostics['acceptance_rate']))

In [ ]:
acc = np.asarray(diagnostics['acceptance_rate'])
print(f"acceptance per chain = {np.round(acc, 3)} (RWMH optimal ~0.234)")
assert np.all((acc > 0.05) & (acc < 0.7)), "acceptance far from RWMH-usable range"

## Whitened trace (cosmology block; truth at 0)

In [ ]:
chain_spec = make_fullshape_spec(
    cosmo_keys=cosmo.param_keys, cosmo_sizes=cosmo.param_sizes, survey_keys=joint_survey_keys,
    n_bins=n_zbins, varied_idx=list(split.nl_idx), analysis_kind='joint')
named_w = packed_samples_to_dict(samples_w, chain_spec, chain_axis=0, draw_axis=1)
cosmo_names = [chain_spec.varied_param_names()[i] for i in cosmo_nl_pos]
cosmo_w = {name: named_w[name] for name in cosmo_names}
fig, _ = plot_trace(cosmo_w, var_names=cosmo_names,
                    truths={name: 0.0 for name in cosmo_names},
                    chain_axis=0, draw_axis=1)

## Physical-space posterior and MCMC-vs-Fisher comparison (cosmology)

In [ ]:
samples_phys = samples_to_physical(samples_w, to_physical)
summary = scalar_summary_packed(samples_phys, chain_spec)

cosmo_flat = np.asarray(samples_phys)[..., cosmo_nl_pos].reshape(-1, N_SHARED)
mcmc_mean = cosmo_flat.mean(axis=0)
mcmc_sigma = cosmo_flat.std(axis=0)
mcmc_cov = np.cov(cosmo_flat, rowvar=False)
fisher_cov = np.asarray(inv(F_pfs_bao_prior_cosmo))
fisher_sig = np.sqrt(np.diag(fisher_cov))

print(f"{'param':>7s} {'fid':>10s} {'MCMC mean':>12s} {'Fisher sig':>12s} {'MCMC sig':>12s} {'ratio':>7s}")
for i, key in enumerate(SHARED_KEYS):
    print(f"{key:>7s} {FIDUCIAL_SHARED[i]:10.5g} {mcmc_mean[i]:12.5g} "
          f"{fisher_sig[i]:12.5g} {mcmc_sigma[i]:12.5g} {mcmc_sigma[i]/fisher_sig[i]:7.2f}")

In [ ]:
# Cosmology corner: MCMC samples (blue) vs Fisher+prior ellipses (red dashed), both at the fiducial.
plt.rcParams.update({"font.size": 10})
fig = plt.figure(figsize=(N_SHARED * 2.4, N_SHARED * 2.4), constrained_layout=True)
handles = [mlines.Line2D([], [], color='royalblue', lw=1.6, label='MCMC'),
           mlines.Line2D([], [], color='crimson', ls='--', lw=1.6, label='Fisher + prior')]
for i in range(N_SHARED):
    for j in range(N_SHARED):
        ax = plt.subplot(N_SHARED, N_SHARED, i * N_SHARED + j + 1)
        if j < i:
            ax.scatter(cosmo_flat[:, j], cosmo_flat[:, i], s=2, alpha=0.15,
                       color='royalblue', rasterized=True)
            plot_contours(F_pfs_bao_prior_cosmo, FIDUCIAL_SHARED, np.array([j, i]),
                          ax=ax, fill=False, color='crimson', ls='--', lw=1.4)
            if i == 1 and j == 0:
                ax.legend(handles=handles, fontsize=7, frameon=False, loc='best')
        elif j == i:
            ax.hist(cosmo_flat[:, i], bins=40, density=True, color='royalblue', alpha=0.4)
            plot_Gaussian(F_pfs_bao_prior_cosmo, FIDUCIAL_SHARED, i, ax=ax, color='crimson', ls='--', lw=1.4)
            ax.set_yticks([])
        else:
            ax.axis('off')
        if i == N_SHARED - 1: ax.set_xlabel(SHARED_LABELS[j])
        if j == 0 and i > 0: ax.set_ylabel(SHARED_LABELS[i])